# Lexsi Track 3 — fresh run (SFT + GRPO)

Clone goes into `lexsi-v2`, so the old broken copy is never used. Run cells top to bottom. Baseline first; then SFT smoke; GRPO real runs are commented for Kaggle.

For Kaggle: replace `%cd /content` with `%cd /kaggle/working` and clone without the `lexsi-v2` suffix, then set `%env CUDA_VISIBLE_DEVICES=0`.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d lexsi-v2 ] && (cd lexsi-v2 && git pull) || git clone https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git lexsi-v2
%cd lexsi-v2

In [ ]:
# 3-6 min. Dependency-conflict warnings come from Colab's preinstalled stack.
!pip install -q -r requirements.txt

**Restart the runtime now** (Runtime > Restart session) so torch loads cleanly.
After restart, continue from the next cell — do not run the install cell again.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from pathlib import Path

script = Path("scripts/train_grpo.py").read_text()
assert "model_init_kwargs" in script, "OLD CODE — run the clone cell again"
assert Path("scripts/train_sft.py").exists(), "OLD CODE — SFT script missing"
print("fp16 fix + SFT script present")
!git log --oneline -1

In [ ]:
# No model needed. Expect: 20/20 success, all scaffold tests passed.
!python data/build_tasks.py
!python scripts/run_oracle_eval.py
!python tests/test_scaffold.py

## Baseline (the left column)

Measured: 2062 s (34 min) for 20 tasks on a Kaggle T4, `success=25%`. Keep `runs/baseline/summary.json`; recover the strict-JSON column with `scripts/score_strict.py`.

In [ ]:
!python scripts/run_baseline.py --model CohereLabs/tiny-aya-global --limit 20

## SFT warm-start (the fix for zero reward)

The 2-step GRPO smoke on the base model logged `frac_reward_zero_std=1`: both completions in every group got the same reward, so there was no advantage to learn from. SFT on oracle traces installs the JSON format and the ledger-copy behaviour first; GRPO then optimises success and efficiency.

Smoke: 20 tasks, 1 epoch. Full run (Kaggle): 100 tasks, 3 epochs, then GRPO from `runs/sft_merged`.

In [ ]:
# Smoke: 20 tasks, 1 epoch. No GPU needed for --check; the train step needs CUDA.
!python scripts/train_sft.py --check
!python scripts/train_sft.py --limit 20 --epochs 1 --out runs/sft_smoke --no-merge

In [ ]:
# Full SFT on Kaggle (commented here). `runs/sft_merged` is the model GRPO loads.
# !python scripts/train_sft.py --tasks data/tasks.jsonl --epochs 3 \
#     --out runs/sft --merge-out runs/sft_merged

## GRPO smoke test (base model)

The base model smoke is a pipeline check only. On Nov 2 it took 786 s for 2 steps and logged `frac_reward_zero_std=1`, so do not launch a long base-model run. Real training starts from `runs/sft_merged` below.

In [ ]:
!python scripts/train_grpo.py --max-steps 2

In [ ]:
# Only after 2 steps pass without OOM.
!python scripts/train_grpo.py --max-steps 10

In [ ]:
# Tight-memory fallback if the T4 still OOMs:
# !python scripts/train_grpo.py --max-steps 10 --max-completion-length 128 --max-steps-per-turn 2

## Real run — use Kaggle 2xT4, not free Colab.
Leave this commented here.

In [ ]:
# Real run (commented): 60 steps from SFT, ~2.5-6 h depending on step time.
# !python scripts/train_grpo.py --model runs/sft_merged \
#     --tasks data/tasks_train.jsonl --out runs/grpo_sft \
#     --max-steps 60 --efficiency

## Tuned evaluation (after a real run)

In [ ]:
# !python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
#     --adapter runs/grpo_sft --tasks data/tasks_eval.jsonl --out runs/tuned_eval

## After the runs
Copy `runs/baseline/summary.json` and `runs/tuned/summary.json` into the table in README.md and docs/hf_article.md.
Upload the adapter, tokenizer, dataset and trajectories.jsonl to Hugging Face.
Deploy `space/app.py` as a GPU Space with `ADAPTER_ID` set.
Submit before Nov 15, 11:59 PM AoE.